In [ ]:
import matplotlib.pyplot as plt
import lovelyplots
plt.style.use('ipynb')

import os
import numpy as np
import mdtraj as md

In [ ]:
true_traj = None

for split in ['train', 'val', 'test']:
    for ref_file in sorted(os.listdir(f'/data/bucket/kleinhej/fast-folding/processed/chignolin/{split}')):
        if not ref_file.endswith('.xtc'):
            continue
        ref_traj = md.load(
            f'/data/bucket/kleinhej/fast-folding/processed/chignolin/{split}/{ref_file}',
            top='/data/bucket/kleinhej/fast-folding/processed/chignolin/filtered.pdb',
        )
        if true_traj is None:
            true_traj = ref_traj
        else:
            true_traj += ref_traj
        
        if true_traj.n_frames >= 100000:
            break
        print(f'{split} {ref_file}: {true_traj.n_frames} frames')

In [ ]:
folded = False
if folded:
    pred_traj = md.load(
        '/homefs/home/daigavaa/boltz/results/fast_folding/boltz_jump_results_s=2.0_d=2.0_f=0.1_chignolin/predictions/chignolin/chignolin_model_0.xtc',
        top='/homefs/home/daigavaa/boltz/results/fast_folding/boltz_jump_results_s=2.0_d=2.0_f=0.1_chignolin/predictions/chignolin/chignolin_model_0.cif',
    )
else:
    pred_traj = md.load(
        '/homefs/home/daigavaa/boltz/results/fast_folding/boltz_jump_results_s=2.0_d=2.0_f=0.1_chignolin_unfolded/predictions/chignolin_unfolded/chignolin_unfolded_model_0.xtc',
        top='/homefs/home/daigavaa/boltz/results/fast_folding/boltz_jump_results_s=2.0_d=2.0_f=0.1_chignolin_unfolded/predictions/chignolin_unfolded/chignolin_unfolded_model_0.cif',
    )

pred_traj

In [ ]:
true_traj

In [ ]:
def align_tops(
    top1: md.Topology, top2: md.Topology
) -> tuple[list[int], list[int]]:
    names1 = [repr(a) for a in top1.atoms]
    names2 = [repr(a) for a in top2.atoms]
    print(names1[:10])
    print(names2[:10])
    intersection = [name for name in names1 if name in names2]

    mask1 = [names1.index(name) for name in intersection]
    mask2 = [names2.index(name) for name in intersection]
    return mask1, mask2

mask_true, mask_pred = align_tops(true_traj.topology, pred_traj.topology)
true_traj_aligned = true_traj.atom_slice(mask_true)
pred_traj_aligned = pred_traj.atom_slice(mask_pred)

true_traj_aligned, pred_traj_aligned

In [ ]:
# Make Ramachandran plots
def plot_ramachandran(traj, title):
    phi_angles = []
    psi_angles = []
    for residue in range(1, traj.n_residues - 1):
        phi = md.compute_phi(traj)[1][:, residue - 1]
        psi = md.compute_psi(traj)[1][:, residue - 1]
        phi_angles.append(phi)
        psi_angles.append(psi)
    phi_angles = np.concatenate(phi_angles)
    psi_angles = np.concatenate(psi_angles)

    plt.figure(figsize=(6, 6))
    plt.hist2d(phi_angles, psi_angles, bins=100, range=[[-np.pi, np.pi], [-np.pi, np.pi]], cmap='Blues')
    plt.colorbar(label='Counts')
    plt.xlabel('Phi')
    plt.ylabel('Psi')
    plt.title(title)
    plt.xlim([-np.pi, np.pi])
    plt.ylim([-np.pi, np.pi])
    plt.grid()
    plt.show()


def plot_ramachandran_pairwise(traj, title):
    phi_indices, phi_angles = md.compute_phi(traj)
    psi_indices, psi_angles = md.compute_psi(traj)
    
    topology = traj.topology
    
    # phi[i] is for residue i+2 (1-indexed), i.e. starts at residue 2
    # psi[i] is for residue i+1 (1-indexed), i.e. starts at residue 1
    # To match: phi[i] should pair with psi[i+1]
    
    n_pairs = min(phi_angles.shape[1], psi_angles.shape[1] - 1)
    n_cols = 4
    n_rows = (n_pairs + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten()
    
    for i in range(n_pairs):
        ax = axes[i]
        phi = phi_angles[:, i]
        psi = psi_angles[:, i + 1]
        
        phi_atoms = [topology.atom(idx) for idx in phi_indices[i]]
        psi_atoms = [topology.atom(idx) for idx in psi_indices[i + 1]]
        
        phi_atom_names = [f"{a.residue.name}{a.residue.resSeq}-{a.name}" for a in phi_atoms]
        psi_atom_names = [f"{a.residue.name}{a.residue.resSeq}-{a.name}" for a in psi_atoms]
        
        # Verify they're for the same residue
        phi_residue = phi_atoms[1].residue  # N of phi
        psi_residue = psi_atoms[0].residue  # N of psi
        
        assert phi_residue == psi_residue, f"Mismatch: phi={phi_residue}, psi={psi_residue}"
        
        print(f"Residue {phi_residue.name}{phi_residue.resSeq}:")
        print(f"  Phi: {' - '.join(phi_atom_names)}")
        print(f"  Psi: {' - '.join(psi_atom_names)}")
        print()
        
        h = ax.hist2d(phi, psi, bins=50, range=[[-np.pi, np.pi], [-np.pi, np.pi]], cmap='Blues')
        ax.set_xlabel(r'$\phi$')
        ax.set_ylabel(r'$\psi$')
        ax.set_title(f'{phi_residue.name}{phi_residue.resSeq}')
        ax.set_xlim([-np.pi, np.pi])
        ax.set_ylim([-np.pi, np.pi])
        ax.grid(True)
    
    for i in range(n_pairs, len(axes)):
        axes[i].set_visible(False)
    
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()


def plot_ramachandran_comparison(true_traj, pred_traj, title):
    phi_indices_true, phi_angles_true = md.compute_phi(true_traj)
    psi_indices_true, psi_angles_true = md.compute_psi(true_traj)
    
    phi_indices_pred, phi_angles_pred = md.compute_phi(pred_traj)
    psi_indices_pred, psi_angles_pred = md.compute_psi(pred_traj)
    
    topology = true_traj.topology
    
    n_pairs = min(phi_angles_true.shape[1], psi_angles_true.shape[1])
    n_cols = 3
    n_rows = n_pairs
    print(f'Plotting {n_pairs} residue pairs')
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    
    for i in range(n_pairs):
        phi_true = phi_angles_true[:, i]
        psi_true = psi_angles_true[:, i]
        
        phi_pred = phi_angles_pred[:, i]
        psi_pred = psi_angles_pred[:, i]
        
        phi_atoms = [topology.atom(idx) for idx in phi_indices_true[i]]
        phi_residue = phi_atoms[1].residue
        
        # True trajectory
        ax = axes[i, 0]
        ax.scatter(phi_true, psi_true, s=1, alpha=0.3, c='blue')
        ax.set_xlabel(r'$\phi$')
        ax.set_ylabel(r'$\psi$')
        ax.set_title(f'{phi_residue.name}{phi_residue.resSeq} - True')
        ax.set_xlim([-np.pi, np.pi])
        ax.set_ylim([-np.pi, np.pi])
        ax.grid(True)
        
        # Predicted trajectory
        ax = axes[i, 1]
        ax.scatter(phi_pred, psi_pred, s=1, alpha=0.3, c='orange')
        ax.set_xlabel(r'$\phi$')
        ax.set_ylabel(r'$\psi$')
        ax.set_title(f'{phi_residue.name}{phi_residue.resSeq} - Predicted')
        ax.set_xlim([-np.pi, np.pi])
        ax.set_ylim([-np.pi, np.pi])
        ax.grid(True)
        
        # Overlay
        ax = axes[i, 2]
        ax.scatter(phi_true, psi_true, s=1, alpha=0.3, c='blue', label='True')
        ax.scatter(phi_pred, psi_pred, s=1, alpha=0.3, c='orange', label='Predicted')
        ax.set_xlabel(r'$\phi$')
        ax.set_ylabel(r'$\psi$')
        ax.set_title(f'{phi_residue.name}{phi_residue.resSeq} - Overlay')
        ax.set_xlim([-np.pi, np.pi])
        ax.set_ylim([-np.pi, np.pi])
        ax.grid(True)
        ax.legend(markerscale=5)
    
    fig.suptitle(title, fontsize=20)
    plt.tight_layout(h_pad=2.0, w_pad=1.0, rect=[0, 0, 1, 0.96])  # rect leaves room for suptitle
    plt.show()


# plot_ramachandran(true_traj_aligned[::1000], 'Ramachandran Plot - True Trajectory')
# plot_ramachandran(pred_traj_aligned, 'Ramachandran Plot - Predicted Trajectory')

# plot_ramachandran_pairwise(true_traj_aligned[::1000], 'Ramachandran Pairwise Plots - True Trajectory')
# plot_ramachandran_pairwise(pred_traj_aligned, 'Ramachandran Pairwise Plots - Predicted Trajectory')

if folded:
    title = 'Chignolin Folded - Ramachandran Plots'
else:
    title = 'Chignolin Unfolded - Ramachandran Plots'
plot_ramachandran_comparison(true_traj_aligned[::100], pred_traj_aligned, 'Chignolin Unfolded - Ramachandran Plots')